# Adding Signals

This notebook covers superposition: signals add sample by sample. That simple rule explains chords, harmonics, constructive and destructive interference, and beat frequencies.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Superposition

If two sources are present at once, the receiver sees their sum. There is no special "mixing" required for simple addition in the time domain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_sum(f1=440.0, a1=0.8, f2=660.0, a2=0.6):
    fs = 44_100
    t = np.arange(0, 0.03, 1 / fs)
    sig1 = a1 * np.cos(2 * np.pi * f1 * t)
    sig2 = a2 * np.cos(2 * np.pi * f2 * t)
    total = sig1 + sig2
    axes[0].clear()
    axes[1].clear()
    plot_waveform(total[:1500], fs=fs, ax=axes[0], title="Sum in Time")
    plot_spectrum(total, fs=fs, ax=axes[1], title="Sum in Frequency")
    axes[1].set_xlim(0, 2000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, normalize(total), rate=fs)

controls = widgets.interactive(
    update_sum,
    f1=float_slider(min_value=100, max_value=1200, step=10, value=440, description="f1"),
    a1=float_slider(min_value=0, max_value=1, step=0.05, value=0.8, description="a1"),
    f2=float_slider(min_value=100, max_value=1600, step=10, value=660, description="f2"),
    a2=float_slider(min_value=0, max_value=1, step=0.05, value=0.6, description="a2"),
)
display(controls, audio_out)


## Build a Chord

Harmonics and chords are just sums of tones. The waveform gets more complex, but the spectrum simply lists the frequencies that are present.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_chord(root=220.0, third_gain=0.7, fifth_gain=0.6, octave_gain=0.4):
    fs = 44_100
    t = np.arange(0, 0.5, 1 / fs)
    freqs = [root, root * 5 / 4, root * 3 / 2, root * 2]
    gains = [1.0, third_gain, fifth_gain, octave_gain]
    chord = sum(g * np.cos(2 * np.pi * f * t) for f, g in zip(freqs, gains))
    axes[0].clear()
    axes[1].clear()
    plot_waveform(chord[:2500], fs=fs, ax=axes[0], title="Chord Waveform")
    plot_spectrum(chord, fs=fs, ax=axes[1], title="Chord Spectrum")
    axes[1].set_xlim(0, 2000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, normalize(chord), rate=fs)

controls = widgets.interactive(
    update_chord,
    root=float_slider(min_value=100, max_value=400, step=5, value=220, description="Root"),
    third_gain=float_slider(min_value=0, max_value=1, step=0.05, value=0.7, description="3rd"),
    fifth_gain=float_slider(min_value=0, max_value=1, step=0.05, value=0.6, description="5th"),
    octave_gain=float_slider(min_value=0, max_value=1, step=0.05, value=0.4, description="8ve"),
)
display(controls, audio_out)


## Key Takeaway

Signal addition is the core rule behind interference and timbre. Once you are comfortable thinking in sums of simple waves, modulation and filtering become much less mysterious.